In [43]:
%pip install pandas
%pip install matplotlib
%pip install numpy

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [10]:
# random points generator
# to be honest, it would have been easier to download some external data... maybe...
!python3 cli/data.py -m 700 -n 1 -se 1 -fi data/data_1D_v1.csv
!python3 cli/data.py -m 700 -n 1 -se 73 -fi data/data_1D_v2.csv
!python3 cli/data.py -m 700 -n 2 -se 50 -fi data/data_2D.csv
!python3 cli/data.py -m 700 -n 5 -se 1 -fi data/data_5D.csv

Successfully generated 700 points in 'data/data_1D_v1.csv'
Successfully generated 700 points in 'data/data_1D_v2.csv'
Successfully generated 700 points in 'data/data_2D.csv'
Successfully generated 700 points in 'data/data_5D.csv'


In [35]:
from pandas import DataFrame
import matplotlib.pyplot as plt

def visualise_data_points_2d(data: DataFrame, image_file_name: str = ""):
  """ Scatter plot for 2D plotting
  """
  plt.style.use("ggplot")
  plt.figure(figsize = (30, 8))
  plt.scatter(x = data["x_0"], y = data["y"], c = "green", s = 4)
  plt.ylabel("y")
  plt.xlabel("x_0")
  
  if image_file_name != "":
    plt.savefig(image_file_name)
    plt.close()

In [34]:
from pandas import DataFrame, read_csv

path: str = "data/data_1D_v1.csv"
image_file_name: str = "assets/data_1D_v1.png"
data: DataFrame = read_csv(path)
visualise_data_points_2d(data = data, image_file_name = image_file_name)

![](assets/data_1D_v1.png)

*~ demonstration plot for 2D scatter for dataset with seed 1*

In [37]:
from pandas import DataFrame, read_csv
from sklearn.model_selection import train_test_split
import numpy as np

data: DataFrame = read_csv("data/data_5D.csv") # testing with a more thick dataset
x: DataFrame = data[data.columns[1:]] 
y: DataFrame = data[["y"]]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.3, random_state = 45)
x_train: np.ndarray = np.array(x_train)
x_test: np.ndarray = np.array(x_test)
y_train: np.ndarray = np.array(y_train)
y_test: np.ndarray = np.array(y_test)

In [40]:
import numpy as np

# collection of some useful evaluation metrics
# sklearn provides their own tested metrics however, it's quite enoyable self-calculating them, though i'd probably stick to sklearn next time
class Metrics():
  @classmethod
  def mean_squared(cls, error: np.ndarray) -> float:
    m: int = error.shape[0]
    return ((error.T @ error) * (1 / m)).item()

  @classmethod
  def r_squared(cls, error: np.ndarray, variance: np.ndarray) -> float:
    return (1 - ((error.T @ error) / (variance.T @ variance))).item()

#### Helpful resources

Making this model has been quite a learning experience; a lot of confusion with numpy matrix shapes definitely... I'll provide the links to the resources that I used to help me along the way and as well as the notebook containing the source code - the notebook that you're reading right now from.  

[1] https://frickp.github.io/matrix-gradient-descent.html - helps with the mathematics of gradient descent as well as helpful tips and tricks to do with matrices   
[2] https://www.gatsby.ucl.ac.uk/teaching/courses/sntn/sntn-2017/resources/Matrix_derivatives_cribsheet.pdf - cheat sheet to the derivatives of matrices when working out m and c

In [38]:
import numpy as np

class Multi_Linear_Regression:
  C: np.ndarray 
  M: np.ndarray 

  def train(self, x_train: np.ndarray, y_train: np.ndarray, lr: float = 1e-5, epoch: int = 1000) -> None:
    m, n = x_train.shape[0], x_train.shape[1]

    self.M: np.ndarray = np.ones((n, 1))
    self.C: np.ndarray = np.ones((1, 1))
    for _ in range(epoch): 
      y_hat: np.ndarray = (x_train @ self.M) + self.C
      error: np.ndarray = y_hat - y_train

      M_der: np.ndarray = (x_train.T @ error) * (2 / m)
      C_der: np.ndarray = (np.ones(m) @ error) * (2 / m)

      self.C: np.ndarray = self.C - (lr * C_der)
      self.M: np.ndarray = self.M - (lr * M_der)
  
  def predict(self, x_test: np.ndarray) -> np.ndarray:
    return (x_test @ self.M) + self.C

In [ ]:
import numpy as np

# training the model
multi_linear_regression: Multi_Linear_Regression = Multi_Linear_Regression()
multi_linear_regression.train(x_train, y_train)

# calculation needed specifically for learning r2
mean: np.ndarray = np.mean(y_test, dtype = np.ndarray)
variance: np.ndarray = y_test - mean

# error between the predicted and the actual
y_hat: np.ndarray = multi_linear_regression.predict(x_test)
error: np.ndarray = y_hat - y_test

print(f"M: \n{multi_linear_regression.M}\n")
print(f"C: \n{multi_linear_regression.C}\n")

print(f"Mean squared error: {Metrics.mean_squared(error)}")
print(f"R2 score: {Metrics.r_squared(error, variance)}")

M: 
[[-0.81062369]
 [ 2.20838306]
 [-5.06963665]
 [-1.92534619]
 [-3.45104472]]

C: 
[[1.10136752]]

Mean squared error: 3315.29626312869
R2 score: 0.9780934911888347
